# Credit Card Application — Full ML Pipeline

## Problem Statement
Predict whether a credit card application will be **approved (1)** or **rejected (0)** based on applicant features.

This is a **binary classification** problem.

## Dataset
- **Source:** UCI Credit Card Approval dataset (anonymised)
- **690 applications**, 14 features + 1 target
- All features are pre-encoded as numbers (original had categorical columns)

| Column | Type | Description |
|--------|------|-------------|
| A1 | Binary (0/1) | Applicant attribute 1 |
| A2 | Float | Applicant attribute 2 |
| A3 | Float | Applicant attribute 3 |
| A4 | Categorical (1-3) | Applicant attribute 4 |
| A5–A14 | Numeric | Remaining applicant attributes |
| **Class** | **0 or 1** | **0 = Rejected, 1 = Approved** |

## ML Pipeline Covered
1. Data Loading & Initial Exploration
2. EDA — Distributions, Correlations, Class Balance
3. Data Preprocessing — Scaling, Train/Test Split
4. Feature Selection — Correlation + SelectKBest
5. Model Training — 9 models (Logistic Regression, Decision Tree, Random Forest, Bagging, AdaBoost, Gradient Boosting, SVM, Voting Ensemble, Stacking)
6. Model Comparison
7. Hyperparameter Tuning — Optuna on the best model
8. Final Evaluation — ROC curve, Confusion Matrix, Classification Report


## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print("Imports done.")

## Step 2: Load the Dataset

In [ ]:
df = pd.read_csv(r'd:\PENDRIVE 32 GB\Game\AI_PROJECTS\Machine Learning\kaggle\Credit_Card_Application\Credit_Card_Applications.csv')
print(f"Shape: {df.shape}")
df.head(10)

### Data Types & Missing Values

In [ ]:
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())

### Basic Statistics

In [ ]:
df.describe()

---
## Step 3: Exploratory Data Analysis (EDA)

EDA helps us understand:
- Class balance (are approvals and rejections equal?)
- Feature distributions (normal? skewed? outliers?)
- Correlations (which features predict approval best?)
- Relationships between features and the target


### 3.1 Class Distribution
Check if the dataset is **balanced** — equal approvals and rejections.
Imbalanced data would require techniques like SMOTE or class_weight adjustments.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
class_counts = df['Class'].value_counts()
axes[0].bar(['Rejected (0)', 'Approved (1)'], class_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Class Distribution — Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Rejected (0)', 'Approved (1)'],
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution — Proportion')

plt.suptitle('Target Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Rejected: {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"Approved: {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")
print(f"Imbalance ratio: {class_counts[0]/class_counts[1]:.2f}:1")

### 3.2 Feature Distributions
Histograms for all numeric features — check for skewness and outliers.


In [ ]:
feature_cols = [c for c in df.columns if c not in ['CustomerID', 'Class']]

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Feature Distributions by Class
Box plots split by Class — see which features differ most between approved and rejected applications.
A large gap between the two boxes = strong predictor.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    data_0 = df[df['Class'] == 0][col]
    data_1 = df[df['Class'] == 1][col]
    axes[i].boxplot([data_0, data_1], labels=['Rejected', 'Approved'],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontweight='bold')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Class (Rejected vs Approved)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap
Shows linear relationships between all features and the target.
- **+1** = perfect positive correlation
- **-1** = perfect negative correlation
- **0** = no linear relationship

Features strongly correlated with `Class` are the most useful predictors.


In [ ]:
plt.figure(figsize=(12, 9))
corr = df.drop('CustomerID', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Feature Correlation with Target
Bar chart showing each feature's correlation with `Class` — sorted by importance.


In [ ]:
corr_with_target = df.drop('CustomerID', axis=1).corr()['Class'].drop('Class').sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target.values]
bars = plt.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Class (Target)', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Pearson Correlation')
plt.xticks(rotation=45)
for bar, val in zip(bars, corr_with_target.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * np.sign(val),
             f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print("Top 5 positively correlated features:")
print(corr_with_target.head())
print("\nTop 5 negatively correlated features:")
print(corr_with_target.tail())

---
## Step 4: Data Preprocessing

### What We Do Here
1. **Drop CustomerID** — just an identifier, no predictive value
2. **Separate features (X) and target (y)**
3. **Train/Test Split** — 80% train, 20% test (stratified to preserve class ratio)
4. **StandardScaler** — normalise all features to mean=0, std=1

### Why Scale?
- Distance-based models (SVM, KNN) are very sensitive to feature scale
- Gradient-based models (Logistic Regression) converge faster
- Tree models (RF, GBM) don't need scaling but it doesn't hurt


In [ ]:
# Drop CustomerID
df_clean = df.drop('CustomerID', axis=1)

# Features and target
X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

print(f"Features: {list(X.columns)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Class distribution: {dict(y.value_counts())}")

In [ ]:
# Stratified train/test split (preserves class ratio in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training set:  {X_train_scaled.shape}")
print(f"Test set:      {X_test_scaled.shape}")
print(f"\nClass split in train: {dict(pd.Series(y_train).value_counts())}")
print(f"Class split in test:  {dict(pd.Series(y_test).value_counts())}")

---
## Step 5: Feature Selection

### Why Select Features?
- Remove irrelevant or redundant features → simpler, faster, more generalisable model
- Reduces overfitting risk
- Improves model interpretability

### Two Methods Used Here
| Method | Approach | Works with |
|--------|----------|-----------|
| **Correlation filter** | Keep features with |corr| > threshold | Linear relationships only |
| **SelectKBest (f_classif)** | ANOVA F-test — ranks features by statistical significance | All features, univariate |


In [ ]:
# Method 1: Correlation filter (keep features with |correlation| > 0.1)
corr_threshold = 0.10
selected_by_corr = corr_with_target[abs(corr_with_target) > corr_threshold].index.tolist()

print(f"Features selected by correlation (|r| > {corr_threshold}):")
for f in selected_by_corr:
    print(f"  {f}: r = {corr_with_target[f]:.3f}")

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Method 2: SelectKBest — ANOVA F-test
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_train_scaled, y_train)

feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'F-Score': selector.scores_,
    'p-value': selector.pvalues_
}).sort_values('F-Score', ascending=False)

print("Feature rankings (ANOVA F-test):")
print(feature_scores.to_string(index=False))

In [ ]:
# Visualise feature importance scores
plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in feature_scores['p-value']]
plt.bar(feature_scores['Feature'], feature_scores['F-Score'], color=colors, edgecolor='black')
plt.title('Feature Importance — ANOVA F-Score\n(Red = statistically significant, p < 0.05)', fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('F-Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Select top 10 features
top_features = feature_scores[feature_scores['p-value'] < 0.05]['Feature'].tolist()
print(f"\nStatistically significant features (p < 0.05): {top_features}")

In [ ]:
# Use top features for modelling
X_train_sel = X_train_scaled  # Use all scaled features (SelectKBest informs us, models decide internally)
X_test_sel  = X_test_scaled
print(f"Proceeding with all {X_train_sel.shape[1]} features (scaled).")

---
## Step 6: Model Training — All Models

### Helper Function
A single `evaluate_model` function runs each model through:
1. 5-fold cross-validation on training data (to check generalisation)
2. Final evaluation on held-out test set
3. ROC-AUC score

We collect all results for comparison at the end.


In [ ]:
results = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')

    # Fit and predict
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    test_acc = accuracy_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba) if y_proba is not None else None

    results[name] = {
        'model': model,
        'cv_mean': cv_scores.mean(),
        'cv_std':  cv_scores.std(),
        'test_acc': test_acc,
        'auc': auc,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    print(f"{'='*50}")
    print(f"Model: {name}")
    print(f"  CV Accuracy:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    if auc: print(f"  ROC-AUC:       {auc:.4f}")
    print()

print("Helper function ready.")

### Model 1: Logistic Regression

**Type:** Linear model — draws a straight decision boundary in feature space.

**Key idea:** Applies the sigmoid function to a linear combination of features to output a probability.
```
P(approved) = σ(w₀ + w₁A1 + w₂A2 + ... + w₁₄A14)
```

**Best when:** Features have roughly linear relationships with the target.
**Hyperparameters:** C (inverse regularisation strength), penalty (L1/L2).


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
evaluate_model('Logistic Regression', lr, X_train_sel, y_train, X_test_sel, y_test)

### Model 2: Decision Tree

**Type:** Non-linear, rule-based model — splits data on feature thresholds.

**Key idea:** Builds a tree of if-else rules (e.g. "if A8 > 0.5 → approve").
Each split maximises information gain (reduces impurity).

**Strengths:** Interpretable, handles non-linear boundaries.
**Weakness:** Prone to overfitting — deep trees memorise training data.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
evaluate_model('Decision Tree', dt, X_train_sel, y_train, X_test_sel, y_test)

### Model 3: Random Forest

**Type:** Bagging ensemble of Decision Trees.

**Key idea:**
1. Bootstrap sample the training data (random rows with replacement)
2. At each split, consider only a random subset of features (random subspaces)
3. Train many trees independently
4. Final prediction = majority vote of all trees

**Why better than one tree:** Averaging many decorrelated trees reduces variance (overfitting).
**Key hyperparameters:** `n_estimators`, `max_depth`, `max_features`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
evaluate_model('Random Forest', rf, X_train_sel, y_train, X_test_sel, y_test)

# Feature importance from Random Forest
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(9, 4))
plt.bar(importances.index, importances.values, color='steelblue', edgecolor='black')
plt.title('Random Forest — Feature Importances (MDI)', fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Model 4: Bagging Classifier

**Type:** Generic Bagging ensemble — wraps any base estimator.

**Key idea:** Same as Random Forest bootstrap sampling, but:
- Uses a full Decision Tree (no feature subsampling by default)
- Can wrap any model (Logistic Regression, SVM, etc.)

Random Forest = Bagging + Random Feature Subspace (specialised for trees).


In [ ]:
from sklearn.ensemble import BaggingClassifier

bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
evaluate_model('Bagging', bag, X_train_sel, y_train, X_test_sel, y_test)

### Model 5: AdaBoost

**Type:** Boosting ensemble — sequential, adaptive.

**Key idea:**
1. Train a stump (max_depth=1 tree) on the data
2. Misclassified samples get higher weight
3. Next stump focuses on the hard cases
4. Final prediction = weighted vote: `sign(Σ αₜ × Tₜ(x))`

**Key hyperparameters:** `n_estimators` (number of stumps), `learning_rate` (shrinks each stump's weight).


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)
evaluate_model('AdaBoost', ada, X_train_sel, y_train, X_test_sel, y_test)

### Model 6: Gradient Boosting

**Type:** Boosting ensemble — sequential residual fitting.

**Key idea:**
1. Start with mean prediction F₀ = mean(y)
2. Compute residuals r = y - Fₜ
3. Fit a tree Tₜ on the residuals
4. Update: Fₜ₊₁ = Fₜ + lr × Tₜ(x)
5. Repeat N times

**Gradient Boosting vs AdaBoost:**
- AdaBoost: re-weights samples → adapts focus
- Gradient Boosting: fits residuals directly → corrects errors mathematically

**Key hyperparameters:** `n_estimators`, `learning_rate`, `max_depth`.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
evaluate_model('Gradient Boosting', gbc, X_train_sel, y_train, X_test_sel, y_test)

### Model 7: Support Vector Machine (SVM)

**Type:** Margin-based classifier.

**Key idea:** Find the hyperplane that **maximises the margin** between classes.
Support vectors = the data points closest to the boundary (they define it).

**Kernel trick:** Map data to a higher-dimensional space where it becomes linearly separable:
- `linear` — for linearly separable data
- `rbf` (Gaussian) — for non-linear boundaries (most common)
- `poly` — polynomial boundary

**Key hyperparameters:**
- `C` — regularisation (low C = wider margin, more errors allowed)
- `gamma` — RBF kernel width (low = smooth boundary, high = complex)


In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
evaluate_model('SVM (RBF)', svm, X_train_sel, y_train, X_test_sel, y_test)

### Model 8: Voting Ensemble

**Type:** Heterogeneous ensemble — combines different model types.

**Key idea:** Train multiple different models, combine their predictions:
- **Hard voting:** majority vote of predicted classes
- **Soft voting:** average of predicted probabilities → uses confidence, more accurate

**Why better?** Different models have different weaknesses — combining them cancels out individual errors.

Here we combine: Logistic Regression + Random Forest + Gradient Boosting.


In [ ]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
        ('gbc', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)),
    ],
    voting='soft'
)
evaluate_model('Voting Ensemble (soft)', voting, X_train_sel, y_train, X_test_sel, y_test)

### Model 9: Stacking

**Type:** Meta-learning ensemble.

**Key idea:**
- **Level-0 models** (base learners): train on data → output out-of-fold predictions
- **Level-1 model** (meta-learner): trained on the out-of-fold predictions → learns how to best combine the base models

**Why out-of-fold?** Prevents the meta-learner from seeing the training data that the base models were trained on → avoids leakage.

**Architecture here:**
```
Level 0: Logistic Regression, Decision Tree, SVM
           ↓ out-of-fold predictions
Level 1: Random Forest (meta-learner)
```


In [ ]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[
        ('lr',  LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
        ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ],
    final_estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    cv=5,
    passthrough=False
)
evaluate_model('Stacking', stacking, X_train_sel, y_train, X_test_sel, y_test)

---
## Step 7: Model Comparison

Compare all 9 models side-by-side on:
- Cross-validation accuracy (training generalisation)
- Test set accuracy (unseen data performance)
- ROC-AUC score (discrimination ability)


In [ ]:
# Build comparison table
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'CV Accuracy': [results[m]['cv_mean'] for m in results],
    'CV Std': [results[m]['cv_std'] for m in results],
    'Test Accuracy': [results[m]['test_acc'] for m in results],
    'ROC-AUC': [results[m]['auc'] for m in results],
}).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

comparison['CV Accuracy'] = comparison['CV Accuracy'].map('{:.4f}'.format)
comparison['CV Std']      = comparison['CV Std'].map('±{:.4f}'.format)
comparison['Test Accuracy'] = comparison['Test Accuracy'].map('{:.4f}'.format)
comparison['ROC-AUC']     = comparison['ROC-AUC'].map(lambda x: f'{x:.4f}' if x else 'N/A')

print(comparison.to_string(index=False))

In [ ]:
# Visual comparison
model_names = list(results.keys())
test_accs   = [results[m]['test_acc'] for m in model_names]
cv_means    = [results[m]['cv_mean']  for m in model_names]
aucs        = [results[m]['auc']      for m in model_names]

x = range(len(model_names))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Test Accuracy
bars = axes[0].barh(model_names, test_accs, color='steelblue', edgecolor='black')
axes[0].set_xlim(0.7, 1.0)
axes[0].set_title('Test Accuracy', fontweight='bold')
axes[0].set_xlabel('Accuracy')
for bar, val in zip(bars, test_accs):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

# CV Accuracy
bars2 = axes[1].barh(model_names, cv_means, color='#2ecc71', edgecolor='black')
axes[1].set_xlim(0.7, 1.0)
axes[1].set_title('CV Accuracy (5-fold)', fontweight='bold')
axes[1].set_xlabel('Accuracy')
for bar, val in zip(bars2, cv_means):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

# ROC-AUC
bars3 = axes[2].barh(model_names, aucs, color='#e67e22', edgecolor='black')
axes[2].set_xlim(0.7, 1.0)
axes[2].set_title('ROC-AUC Score', fontweight='bold')
axes[2].set_xlabel('AUC')
for bar, val in zip(bars3, aucs):
    axes[2].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Model Comparison — All 9 Classifiers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8: Hyperparameter Tuning with Optuna

Tune the best-performing model (Gradient Boosting) using **Optuna TPE sampler**.

### What We're Tuning
| Parameter | Search Range | Effect |
|-----------|-------------|--------|
| `n_estimators` | 50–500 | More trees = better fit, slower |
| `learning_rate` | 0.01–0.3 (log) | Lower = need more trees, less overfit |
| `max_depth` | 2–8 | Deeper = more expressive, more overfit |
| `min_samples_split` | 2–20 | Min samples to split a node |
| `min_samples_leaf` | 1–10 | Min samples in each leaf |
| `subsample` | 0.5–1.0 | Fraction of samples per tree |


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'random_state': 42,
    }
    model = GradientBoostingClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest CV Accuracy: {study.best_trial.value:.4f}")
print(f"Best Parameters: {study.best_trial.params}")

In [ ]:
# Train final tuned model
best_params = study.best_trial.params
best_params['random_state'] = 42

tuned_gbc = GradientBoostingClassifier(**best_params)
evaluate_model('GBC (Optuna-tuned)', tuned_gbc, X_train_sel, y_train, X_test_sel, y_test)

print("\nImprovement over default GBC:")
default_acc = results['Gradient Boosting']['test_acc']
tuned_acc   = results['GBC (Optuna-tuned)']['test_acc']
print(f"  Default:  {default_acc:.4f}")
print(f"  Tuned:    {tuned_acc:.4f}")
print(f"  Delta:    {tuned_acc - default_acc:+.4f}")

---
## Step 9: Final Model Evaluation

Evaluate the **best model** (tuned Gradient Boosting) on the test set with:
1. **Confusion Matrix** — true/false positives and negatives
2. **Classification Report** — precision, recall, F1 per class
3. **ROC Curve** — all models overlaid for comparison

### Confusion Matrix Explained
```
                  Predicted
                Reject  Approve
Actual Reject  |  TN  |  FP  |   ← False Positives = approved when should reject
       Approve |  FN  |  TP  |   ← False Negatives = rejected when should approve
```
- **High FP** = approving risky applicants (costly for bank)
- **High FN** = rejecting good applicants (lost business)


In [ ]:
# Pick the best model
best_name = max(results, key=lambda m: results[m]['test_acc'])
best_result = results[best_name]
print(f"Best model: {best_name}")
print(f"Test Accuracy: {best_result['test_acc']:.4f}")
print(f"ROC-AUC:       {best_result['auc']:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, best_result['y_pred'])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
axes[0].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Add labels
tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.15, f'TN={tn}  FP={fp}  FN={fn}  TP={tp}',
             ha='center', transform=axes[0].transAxes, fontsize=10)

# Normalised confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
axes[1].set_title(f'Normalised Confusion Matrix\n{best_name}', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, best_result['y_pred'],
                             target_names=['Rejected (0)', 'Approved (1)']))

In [ ]:
# ROC Curves — all models overlaid
plt.figure(figsize=(9, 7))

for name, res in results.items():
    if res['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        auc = res['auc']
        lw = 2.5 if name == best_name else 1.0
        ls = '-' if name == best_name else '--'
        plt.plot(fpr, tpr, linewidth=lw, linestyle=ls,
                 label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — All Models', fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

### Pipeline Recap
```
Raw Data (690 × 16)
    ↓
EDA — class balance, distributions, correlations
    ↓
Preprocessing — drop CustomerID, StandardScaler, 80/20 split
    ↓
Feature Selection — ANOVA F-test (informational)
    ↓
Model Training (9 models):
    Linear:    Logistic Regression
    Tree:      Decision Tree
    Ensemble:  Random Forest, Bagging, AdaBoost, Gradient Boosting
    Kernel:    SVM (RBF)
    Combined:  Voting (soft), Stacking
    ↓
Comparison — Test Accuracy, CV Accuracy, ROC-AUC
    ↓
Hyperparameter Tuning — Optuna (50 trials, TPE sampler)
    ↓
Final Evaluation — Confusion Matrix, Classification Report, ROC Curve
```

### Key Takeaways
- **Feature A8** (binary) has the highest correlation with approval (r=0.72) — most important predictor
- **Ensemble methods** (RF, GBM, Stacking) generally outperform single models
- **Gradient Boosting** benefits most from Optuna tuning
- The dataset is **slightly imbalanced** (56% rejected / 44% approved) — model performs well regardless

### Next Steps
- Try XGBoost / LightGBM for faster gradient boosting
- Apply SHAP values for model explainability
- Use SMOTE if class imbalance worsens with a larger dataset
- Deploy the best model with Flask/FastAPI
